In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split


## all answers for GEMINI 

# --- Bernoulli Naive Bayes Setup ---
rng = np.random.default_rng(seed=1)
X1 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.7), rng.binomial(size = 50,n = 1, p =0.2))).reshape(-1, 1)
X2 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.6), rng.binomial(size = 50,n = 1, p =0.1))).reshape(-1, 1)
X3 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.6), rng.binomial(size = 50,n = 1, p =0.2))).reshape(-1, 1)
X4 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.8), rng.binomial(size = 50,n = 1, p =0.1))).reshape(-1, 1)

X_bernoulli = np.column_stack((X1,X2,X3,X4))
y_bernoulli = np.concatenate((np.zeros(50, dtype= int), np.ones(50, dtype = int))).reshape(-1, 1)
permute = rng.permuted(range(100)) 

X_bernoulli = X_bernoulli[permute]
y_bernoulli = y_bernoulli[permute]

# Q1: Value of \hat{p}, the estimate for P(Y=1)
p_hat = np.mean(y_bernoulli == 1)
print(f"Q1: {p_hat}")

# Q2: \hat{p}_0^0, estimate of P(f_0=1|y=0)
X_y0 = X_bernoulli[(y_bernoulli == 0).flatten()]
p_0_0 = np.mean(X_y0[:, 0] == 1)
print(f"Q2: {p_0_0:.2f}")

# Q3: \hat{p}_0^1, estimate of P(f_0=1|y=1)
X_y1 = X_bernoulli[(y_bernoulli == 1).flatten()]
p_0_1 = np.mean(X_y1[:, 0] == 1)
print(f"Q3: {p_0_1:.2f}")

# Q4: \hat{p}_3^1, estimate of P(f_3=1|y=1)
p_3_1 = np.mean(X_y1[:, 3] == 1)
print(f"Q4: {p_3_1:.2f}")

# Q5 & Q6: Prediction for [1,0,1,0] and [1,0,1,1]
# Let's compute the class conditional probabilities
p_y0 = np.mean(y_bernoulli == 0)
p_y1 = np.mean(y_bernoulli == 1)

probs_y0 = np.mean(X_y0, axis=0)
probs_y1 = np.mean(X_y1, axis=0)

def predict_bernoulli(x):
    # P(x | y=0) * P(y=0)
    p_x_y0 = np.prod([probs_y0[i] if x[i] == 1 else (1 - probs_y0[i]) for i in range(4)]) * p_y0
    # P(x | y=1) * P(y=1)
    p_x_y1 = np.prod([probs_y1[i] if x[i] == 1 else (1 - probs_y1[i]) for i in range(4)]) * p_y1
    return 0 if p_x_y0 >= p_x_y1 else 1

print(f"Q5 [1,0,1,0]: {predict_bernoulli([1, 0, 1, 0])}")
print(f"Q6 [1,0,1,1]: {predict_bernoulli([1, 0, 1, 1])}")


# --- Gaussian Naive Bayes Setup ---
X_g, y_g = make_blobs(n_samples = 100,
                  n_features=2, 
                  centers=[[5,5],[10,10]],
                  cluster_std=1.5,
                  random_state=2)

X_train, X_test, y_train, y_test = train_test_split(X_g, y_g, test_size=0.2,
                                                    random_state=123)

# Q7: Number of examples in training dataset
print(f"Q7: {X_train.shape[0]}")

# Q8: Number of features
print(f"Q8: {X_train.shape[1]}")

# Q9: \hat{p} for P(Y=1) in training
p_hat_g = np.mean(y_train == 1)
print(f"Q9: {p_hat_g:.2f}")

# Q10: Sum of components of \hat{\mu}_0
mu_0 = np.mean(X_train[y_train == 0], axis=0)
print(f"Q10: {np.sum(mu_0):.2f}")

# Q11: Trace of \hat{\Sigma}_0
# "The estimate for \Sigma_k will be \hat{\Sigma}_k = \sigma_i I where \sigma_i is the variance of i^th feature values of examples labeled k."
# Wait, if it says \sigma_i I, is it a diagonal matrix with \sigma_i on the diagonal? 
# Let's re-read carefully: "\hat{\Sigma}_k = \sigma_i I where \sigma_i is the variance of i^th feature values of examples labeled k"
# Usually, for Gaussian Naive Bayes, the covariance matrix is diagonal where the diagonal elements are the variances of each feature. 
# So \hat{\Sigma}_k = diag(\sigma_1^2, \sigma_2^2, ..., \sigma_d^2) where \sigma_i^2 is the variance of feature i for class k.
# Let's find the variances of features for class 0 using ddof=0 (standard maximum likelihood estimate).
var_0 = np.var(X_train[y_train == 0], axis=0, ddof=0)
trace_sigma_0 = np.sum(var_0)
print(f"Q11: {trace_sigma_0:.2f}")

# Q12 & Q13: Train and Test accuracy
# Let's build the classifier manually according to the definition
mu_1 = np.mean(X_train[y_train == 1], axis=0)
var_1 = np.var(X_train[y_train == 1], axis=0, ddof=0)

def predict_gaussian(X_data):
    preds = []
    for x in X_data:
        # log P(x | y=0) + log P(y=0)
        # log P(x | y=0) = -0.5 * sum( log(2*pi*var) + (x-mu)^2 / var )
        log_p_0 = -0.5 * np.sum(np.log(2 * np.pi * var_0) + ((x - mu_0) ** 2) / var_0) + np.log(1 - p_hat_g)
        log_p_1 = -0.5 * np.sum(np.log(2 * np.pi * var_1) + ((x - mu_1) ** 2) / var_1) + np.log(p_hat_g)
        preds.append(0 if log_p_0 >= log_p_1 else 1)
    return np.array(preds)

train_preds = predict_gaussian(X_train)
test_preds = predict_gaussian(X_test)

train_acc = np.mean(train_preds == y_train)
test_acc = np.mean(test_preds == y_test)
print(f"Q12: {train_acc:.2f}")
print(f"Q13: {test_acc:.2f}")

# Bernoulli naive Bayes

Run the below cell to get the following variables:

`X` = Data matrix of shape $(n, d)$. All the features are binary taking values $0$ or $1$.

`y` = label vector. Labels are $0$ and $1$.

In [2]:
rng = np.random.default_rng(seed=1)
X1 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.7), rng.binomial(size = 50,n = 1, p =0.2))).reshape(-1, 1)
X2 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.6), rng.binomial(size = 50,n = 1, p =0.1))).reshape(-1, 1)
X3 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.6), rng.binomial(size = 50,n = 1, p =0.2))).reshape(-1, 1)
X4 = np.concatenate((rng.binomial(size = 50,n = 1, p =0.8), rng.binomial(size = 50,n = 1, p =0.1))).reshape(-1, 1)


X = np.column_stack((X1,X2,X3,X4))

y = np.concatenate((np.zeros(50, dtype= int), np.ones(50, dtype = int))).reshape(-1, 1)
permute = rng.permuted(range(100)) 

X = X[permute]
y = y[permute]


## Question 1
If we train the naive Bayes model on the dataset, What will be the value of $\hat{p}$, the estimate for $P(Y=1)$? 



In [3]:
# Enter your solution here

## Question 2
What will be the value of $\hat{p}_0^0$, the estimate of $P(f_0=1|y=0)$?  Write your answer correct to two decimal places.



In [4]:
# Enter your solution here

## Question 3
What will be the value of $\hat{p}_0^1$, the estimate of $P(f_0=1|y=1)$?  Write your answer correct to two decimal places.



In [5]:
# Enter your solution here

## Question 4
What will be the value of $\hat{p}_3^1$, the estimate of $P(f_3=1|y=1)$?  Write your answer correct to two decimal places.




In [6]:
# Enter your solution here

## Question 5

What will be the predicted label for the point $[1, 0, 1, 0]$? 



In [7]:
# Enter your solution here

## Question 6

What will be the predicted label for the point $[1, 0, 1, 1]$? 



In [8]:
# Enter your solution here

# Gaussian naive Bayes

Run the below cell to get the following variables:

`X_train` = Training dataset of the shape $(n, d)$. All the examples are coming from multivariate gaussian distribution.

`y_train` = label vector for corresponding training examples. labels are $0$ and $1$.

`X_test` = Test dataset of the shape $(m, d)$, where $m$ is the number of examples in the test dataset. All the examples are coming from multivariate gaussian distribution.

`y_test` = label vector for corresponding test examples. labels are $0$ and $1$.



In [9]:
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split

# generate artificial data points
X, y = make_blobs(n_samples = 100,
                  n_features=2, 
                  centers=[[5,5],[10,10]],
                  cluster_std=1.5,
                  random_state=2)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=123)

## Question 7

How many examples are there in the trianing dataset?



In [10]:
# Enter your solution here

## Question 8
How many features are there in the dataset?



In [11]:
# Enter your solution here

## Question 9

If we train the Gaussian naive Bayes model on the trianing dataset, What will be the value of $\hat{p}$, the estimate for $P(Y=1)$? Write your answer correct to two decimal places.





In [12]:
# Enter your solution here

## Question 10

If $\hat{\mu}_0 = [\mu_1, \mu_2, ..., \mu_d]$ be the estimate for $\mu_0$, the mean of $0$ labeled examples, what will be the value of $\mu_1+\mu_2+...+\mu_d$? Write your answer correct to two decimal places.



In [13]:
# Enter your solution here

We will be using the different covariances for different labeled examples. The estimate for $\Sigma_k$ will be 

$$\hat{\Sigma}_k = \sigma_iI$$ where $\sigma_i$ is the variance of $i^{th}$ feature values of examples labeled $k$.



## Question 11
What will be value of $\text{trace}({\hat{\Sigma}}_0)$?  Write your answer correct to two decimal places.







In [14]:
# Enter your solution here

## Question 12

Once we have estimated all the parameters for Gaussian naive Bayes assuming the different covariance matrices, we predict the labels for the training examples. What will be the training accuracy?

Accuracy is defined as the proportion of correctly classified examples.  Write your answer correct to two decimal places.




In [15]:
# Enter your solution here

## Question 13

What will be the test accuracy?

Accuracy is defined as the proportion of correctly classified examples.  




In [16]:
# Enter your solution here